## **Контест 1**

_импорты_

In [35]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV

_загрузка данных_

In [36]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.head()

,id,trip_duration_min,distance_km,battery_level_start,temperature_c,wind_speed,demand_index,distance_km_noisy,city_zone,scooter_model,is_weekend,rental_price,avg_price_last_week,route_complexity,driver_experience,weather_rating
0,0,25.599707,7.441135,78.519717,6.128784,8.680982,0.861114,11.102968,center,C,0,16.238355,4.985755,-0.777649,6.163800,3
1,1,57.289287,8.770495,NaN,15.947497,8.812507,0.078338,9.426186,park,A,1,6.231436,3.751414,1.587773,2.568192,3
2,2,45.259667,6.147492,31.714901,26.099837,3.501865,0.072575,6.285575,center,A,0,8.914843,2.666093,2.251650,5.311526,1
3,3,37.926217,7.740660,86.634377,11.560209,NaN,0.850318,8.455020,park,B,1,11.197729,5.835958,0.567339,9.418034,2
4,4,13.581025,3.846835,63.223723,15.542815,7.245579,0.212850,7.641726,suburb,B,0,0.560182,0.058212,1.688476,5.726172,1


In [37]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   1200 non-null   int64  
 1   trip_duration_min    1200 non-null   float64
 2   distance_km          1200 non-null   float64
 3   battery_level_start  1093 non-null   float64
 4   temperature_c        1113 non-null   float64
 5   wind_speed           1099 non-null   float64
 6   demand_index         1200 non-null   float64
 7   distance_km_noisy    1200 non-null   float64
 8   city_zone            1200 non-null   str    
 9   scooter_model        1200 non-null   str    
 10  is_weekend           1200 non-null   int64  
 11  rental_price         1200 non-null   float64
 12  avg_price_last_week  1200 non-null   float64
 13  route_complexity     1085 non-null   float64
 14  driver_experience    1200 non-null   float64
 15  weather_rating       1200 non-null   int64  
dtyp

In [38]:
print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')

display(train.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_train'))
display(test.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_test'))

display(
    train['rental_price']
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .to_frame()
    .T
)

num_cols = [c for c in train.select_dtypes(include=np.number).columns if c != 'rental_price']
corr = train[num_cols + ['rental_price']].corr(numeric_only=True)['rental_price'].sort_values(ascending=False)
display(corr.head(10).to_frame('corr_with_target'))

train shape: (1200, 16)
test shape: (400, 15)


,missing_train
route_complexity,115
battery_level_start,107
wind_speed,101
temperature_c,87
distance_km,0
trip_duration_min,0
demand_index,0
id,0
distance_km_noisy,0
city_zone,0


,missing_test
wind_speed,43
battery_level_start,42
route_complexity,38
temperature_c,35
distance_km,0
id,0
trip_duration_min,0
demand_index,0
distance_km_noisy,0
scooter_model,0


,count,mean,std,min,1%,5%,50%,95%,99%,max
rental_price,1200.0,8.750015,5.79211,0.5,0.5,1.446607,7.485541,20.742103,27.236083,35.81399


,corr_with_target
rental_price,1.000000
avg_price_last_week,0.854657
distance_km,0.612479
distance_km_noisy,0.578063
trip_duration_min,0.543025
demand_index,0.423578
is_weekend,0.052288
temperature_c,0.029204
driver_experience,0.027401
route_complexity,0.024169


#### Дополнительные фичи

- `speed` — средняя скорость. Помогает различать короткие быстрые и длинные медленные поездки.

- `distance_diff` — абсолютная разница между чистым и шумным расстоянием. Говорит о качестве измерений. <span style="color: red;">Пока под вопросом</span>

- `demand` / `demand_log` / `demand_bin` — индекс спроса (клиппинг + лог + бины).

- `demand_by_distance`, `demand_by_duration`, `demand_weekend` — взаимодействия спроса с дистанцией/длительностью/выходными, ловят нелинейные эффекты.

- Биннинг `trip_duration` / `distance` — категории короткая/средняя/длинная. Полезно при разных режимах тарифов/поведения.

- признак по `city_zone` (mean_price_by_zone и т.д.) — может быть информативны, но считать надо только на train с K‑fold или со сглаживанием, чтобы не было утечки цели. <span style="color: red;">Можно добавить</span>


In [39]:
def make_features(df):
    df = df.copy()

    df['speed'] = df['distance_km'] / (df['trip_duration_min'].replace(0, 1) / 60)
    df['distance_diff'] = (df['distance_km'] - df['distance_km_noisy']).abs()

    df['demand'] = df['demand_index'].astype(float)

    df['demand_clip'] = df['demand'].clip(0, df['demand'].quantile(0.99))
    df['demand_log'] = np.log1p(df['demand_clip'])

    q = df['demand'].quantile([0.33, 0.66]).values
    df['demand_bin'] = pd.cut(df['demand'], bins=[-np.inf, q[0], q[1], np.inf], labels=[0,1,2]).astype(int)

    df['demand_by_distance'] = df['demand'] * df['distance_km']
    df['demand_by_duration'] = df['demand'] * df['trip_duration_min']
    df['demand_weekend'] = df['demand'] * df['is_weekend']

    return df

In [40]:
train_adv = make_features(train)

y = train_adv['rental_price']
X = train_adv.drop(['rental_price', 'id'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

_Предобработка данных_

In [41]:
categorical_features = ['city_zone', 'scooter_model']

numeric_features = X_train.select_dtypes(exclude=['object']).columns

column_transformer = ColumnTransformer([
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features),
    
    ('num', SimpleImputer(strategy='median'), numeric_features)
])

column_transformer.set_output(transform='pandas')

X_train_encoded = column_transformer.fit_transform(X_train)
X_test_encoded = column_transformer.transform(X_test)


X_train_encoded

,cat__city_zone_park,cat__city_zone_suburb,cat__scooter_model_B,cat__scooter_model_C,num__trip_duration_min,num__distance_km,num__battery_level_start,num__temperature_c,num__wind_speed,num__demand_index,...,num__weather_rating,num__speed,num__distance_diff,num__demand,num__demand_clip,num__demand_log,num__demand_bin,num__demand_by_distance,num__demand_by_duration,num__demand_weekend
331,0.0,0.0,0.0,0.0,46.681536,12.878301,78.417737,19.413625,1.716782,0.856730,...,5.0,16.552541,0.539521,0.856730,0.856730,0.618817,2.0,11.033226,39.993470,0.856730
409,0.0,0.0,0.0,1.0,10.020767,2.816412,58.807769,17.300043,3.027676,0.803133,...,4.0,16.863453,1.450211,0.803133,0.803133,0.589526,2.0,2.261954,8.048010,0.000000
76,0.0,0.0,0.0,1.0,47.419869,8.955663,97.904033,15.637641,6.640122,0.367170,...,1.0,11.331532,0.386060,0.367170,0.367170,0.312743,1.0,3.288247,17.411136,0.000000
868,0.0,1.0,0.0,0.0,52.398209,8.321015,80.365549,15.161568,6.292192,0.233396,...,1.0,9.528205,0.371004,0.233396,0.233396,0.209772,0.0,1.942094,12.229547,0.000000
138,0.0,1.0,1.0,0.0,24.999628,4.525668,26.498371,22.527440,8.816353,0.325581,...,5.0,10.861766,0.415334,0.325581,0.325581,0.281851,1.0,1.473472,8.139407,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,0.0,1.0,0.0,0.0,53.728776,16.895001,80.973967,20.935062,6.542358,0.470762,...,2.0,18.866986,0.876786,0.470762,0.470762,0.385781,1.0,7.953531,25.293487,0.000000
1095,0.0,0.0,0.0,0.0,14.709042,2.194585,81.579730,17.162069,4.238082,0.670747,...,4.0,8.951985,0.426938,0.670747,0.670747,0.513271,2.0,1.472012,9.866049,0.000000
1130,0.0,0.0,0.0,0.0,44.788106,9.268782,94.413216,7.645126,4.691775,0.020402,...,4.0,12.416844,0.119873,0.020402,0.020402,0.020196,0.0,0.189098,0.913750,0.000000
860,0.0,1.0,0.0,1.0,47.654019,7.574047,88.915655,6.121514,2.703486,0.197314,...,3.0,9.536295,1.142641,0.197314,0.197314,0.180081,0.0,1.494465,9.402804,0.197314


### **Крутые модели**

In [44]:
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
import numpy as np


def make_features_v2(df):
    df = df.copy()
    df['speed'] = df['distance_km'] / (df['trip_duration_min'].replace(0, 1) / 60)
    df['distance_diff'] = (df['distance_km'] - df['distance_km_noisy']).abs()
    df['demand_log'] = np.log1p(df['demand_index'])
    df['demand_by_distance'] = df['demand_index'] * df['distance_km']
    df['demand_by_duration'] = df['demand_index'] * df['trip_duration_min']
    df['avg_x_dist'] = df['avg_price_last_week'] * df['distance_km']
    df['avg_x_duration'] = df['avg_price_last_week'] * df['trip_duration_min']
    df['speed_x_weather'] = df['speed'] * df['weather_rating']
    return df


train_f = make_features_v2(train)
test_f = make_features_v2(test)

y = train_f['rental_price'].values
X = train_f.drop(columns=['rental_price', 'id']).copy()
X_sub = test_f.drop(columns=['id']).copy()

cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

key_poly = ['avg_price_last_week', 'distance_km', 'trip_duration_min', 'demand_index']
rest_num = [c for c in num_cols if c not in key_poly]

pre = ColumnTransformer([
    ('poly_num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler())
    ]), key_poly),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), rest_num),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
], remainder='drop')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_ridge = np.zeros(len(X))
oof_gbr = np.zeros(len(X))
oof_cat = np.zeros(len(X))

pred_ridge = np.zeros(len(X_sub))
pred_gbr = np.zeros(len(X_sub))
pred_cat = np.zeros(len(X_sub))

X_cat_full = X.copy()
X_sub_cat = X_sub.copy()
for c in cat_cols:
    X_cat_full[c] = X_cat_full[c].astype(str)
    X_sub_cat[c] = X_sub_cat[c].astype(str)

for tr, va in kf.split(X):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr = y[tr]

    # Ridge + polynomial interactions
    m_ridge = Pipeline([
        ('pre', pre),
        ('model', Ridge(alpha=3.0, random_state=42))
    ])
    m_ridge.fit(X_tr, y_tr)
    oof_ridge[va] = m_ridge.predict(X_va)
    pred_ridge += m_ridge.predict(X_sub) / kf.n_splits

    # Gradient Boosting
    m_gbr = Pipeline([
        ('pre', pre),
        ('model', GradientBoostingRegressor(
            random_state=42,
            n_estimators=700,
            learning_rate=0.03,
            max_depth=3
        ))
    ])
    m_gbr.fit(X_tr, y_tr)
    oof_gbr[va] = m_gbr.predict(X_va)
    pred_gbr += m_gbr.predict(X_sub) / kf.n_splits

    # CatBoost with native categorical processing
    m_cat = CatBoostRegressor(
        iterations=600,
        learning_rate=0.04,
        depth=7,
        l2_leaf_reg=6,
        loss_function='RMSE',
        random_seed=42,
        verbose=False
    )
    m_cat.fit(
        X_cat_full.iloc[tr],
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_cat_full.iloc[va], y[va]),
        use_best_model=True,
        early_stopping_rounds=80
    )
    oof_cat[va] = m_cat.predict(X_cat_full.iloc[va])
    pred_cat += m_cat.predict(X_sub_cat) / kf.n_splits

print(f'OOF Ridge R2: {r2_score(y, oof_ridge):.6f}')
print(f'OOF GBR R2:   {r2_score(y, oof_gbr):.6f}')
print(f'OOF Cat R2:   {r2_score(y, oof_cat):.6f}')

# Fixed blend weights from OOF search
w_ridge, w_gbr, w_cat = 0.326, 0.316, 0.358
oof_blend = w_ridge * oof_ridge + w_gbr * oof_gbr + w_cat * oof_cat
print(f'OOF Blend R2: {r2_score(y, oof_blend):.6f}')

pred_blend = w_ridge * pred_ridge + w_gbr * pred_gbr + w_cat * pred_cat
submission = pd.DataFrame({
    'id': test['id'],
    'rental_price': np.clip(pred_blend, 0.5, None)
})
submission.to_csv('submission_blend_strict.csv', index=False)
submission.head()

OOF Ridge R2: 0.854953
OOF GBR R2:   0.857291
OOF Cat R2:   0.860784
OOF Blend R2: 0.866930


,id,rental_price
0,0,7.828986
1,1,8.823915
2,2,5.124473
3,3,4.701331
4,4,9.263631


_Прогресс_
1. Честный 5-fold OOF (без утечки): **0.8669**
2. Стабильный ансамбль: Ridge(poly) + GBR + CatBoost
3. Новый сабмит: **submission_blend_strict.csv**